# Tutorial: WorldQuant Brain — Bigdata.com Volume API

Get aggregated document and chunk counts over time for a query—without retrieving documents—so **you** can measure coverage, sentiment trends, and when to run heavier Search API calls. This tutorial shows how to use the Volume API to get **aggregated document and chunk counts over time** for a given query, without retrieving the actual documents.

**What you'll learn**
- Call the Volume API with a text query and time (and optional entity) filters
- Interpret the response: totals and per-day volume (documents, chunks, sentiment)
- Use volume data to visualize how coverage and sentiment evolve over time (e.g. for theme or entity analysis)

**Why Bigdata.com Volume API**
- Lightweight way to measure *how much* content matches a query over time (no document payload)
- Same semantic query and filters as Search API (text, timestamp, entity), so you can align volume trends with document retrieval
- Useful for backtesting (e.g. 2021–2022), thematic coverage, and entity-level volume evolution

**Example use cases:** theme volume over time, entity mention trends, sentiment evolution, deciding when to run heavier Search API calls.

**Signals:** Volume over time shows where you have enough data to build a stable signal; use it to decide when and where to run heavier Search API calls.

**In this notebook**
1. **Volume API** — basic call, options (query/filters/ranking), and response structure.
2. **Volume evolution visualization** — per-day volume and sentiment plots, entity filter.

**Prerequisites:** `.env` file with `BRAIN_EMAIL` and `BRAIN_PASSWORD` (see `.env.example`). For access, see the challenge or [WorldQuant Brain](https://www.worldquantbrain.com/) documentation.

**Time:** about 15–20 minutes.

**Endpoint documentation:** [Volume API reference](https://docs.bigdata.com/api-reference/search/get-volume-data)

In [ ]:
from session import BrainSession
from print_helpers import print_volume_results

## Setup & Configuration

In [ ]:
# Load credentials from .env file
import os
from dotenv import load_dotenv
load_dotenv()

# WorldQuant Brain API Configuration
API_BASE_URL = "https://api.worldquantbrain.com"
EMAIL = os.getenv("BRAIN_EMAIL")
PASSWORD = os.getenv("BRAIN_PASSWORD")

# API Endpoints
VOLUME_ENDPOINT = f"{API_BASE_URL}/bigdata/v1/search/volume"

session = BrainSession(
    base_url=API_BASE_URL,
    email=EMAIL,
    password=PASSWORD
)

## Key considerations before starting

The provider APIs do not have full coverage for parameter validation—double-check your request against the [API reference](https://docs.bigdata.com/api-reference) to avoid issues from typos or invalid values. For historical/backtest windows (e.g. 2021–2022), set `freshness_boost` to `0` so results are not biased toward recent data.

In [ ]:
# Date range for examples
START_DATE = "2021-01-01"
END_DATE = "2022-12-30"
TEXT = "Global semiconductor shortage impacts"
ENTITY_ID = "D8442A"  # Apple Inc

## 1. Volume API

The Volume API returns **aggregated document and chunk counts per day** for a given query, without returning the actual documents. You use the same **semantic text query** and **filters** (timestamp, entity) as the Search API, so volume trends align with what you would get from search.

We start with a basic example: daily volume in 2021 for content that semantically matches a query about semiconductor shortage. Then we show how to filter by entity (e.g. Apple) and how to interpret the response.

**Documentation:** [Volume API reference](https://docs.bigdata.com/api-reference/search/get-volume-data)

### Options in the Volume API: Query, Filters, and Ranking

Before we run the query, the options below (query, filters, ranking) are worth reviewing; in particular set `auto_enrich_filters` to `False` and `freshness_boost` to `0` for historical use. The Volume API gives you fine-grained control over *what* gets counted (**query** and **filters**) and *how* recency is applied (**ranking parameters**). This section explains each option in detail so you can build precise, reproducible volume queries—especially important for backtesting and point-in-time research.

You can shape what gets counted in several ways:

- **By text:** Semantic search query. The API counts documents and chunks that match this text by meaning (same as Search API). Use a short phrase or theme (e.g. "Global semiconductor shortage impacts").
- **By time:** Restrict to a time window with the `timestamp` filter using `start` and `end` in ISO 8601 format. Only content within this window is counted. **Required for meaningful time-series volume.**
- **By entity:** Include only documents that mention specific organizations, people, or other entities using the `entity` filter with `any_of`, `all_of`, or `none_of`. Entity IDs are used (e.g. from a PiT Russell 3000 mapping or entity lookup). Optional `search_in`: **HEADLINE**, **BODY**, or **ALL** (defaults to **ALL**).
- **By keyword:** Target documents that contain certain terms using the `keyword` filter with `any_of` / `all_of` / `none_of`. Same optional `search_in` as `entity`.
- **By sentiment:** Keep only chunks whose overall sentiment falls in a given range (e.g. only negative or only positive).
- **By source:** Limit results to specific publishers using the `source` filter (source ID).
- **By document type:** For the challenge, only `news` is available; use `document_type` to restrict to news documents.
- **Recommendations:** Set `auto_enrich_filters` to `False` so entities are not auto-extracted from the text; set `freshness_boost` to `0` for historical/backtest use so recency does not bias the counts.

All of these options can be combined; the API applies them together (e.g. timestamp AND entity AND keyword). The same filters are available on the **Search** and **Co-mentions** endpoints (each endpoint returns different payloads—documents, volume counts, or co-mentioned entities—but filter semantics are shared). Below is a concise reference with didactic notes.

---

### Query and filters (what gets counted)

| Parameter / Filter | Description |
|-------------------|-------------|
| `text` | Semantic search query. The API counts documents and chunks that match this text by meaning (same as Search API). Use a short phrase or theme. |
| `auto_enrich_filters` | When set to `True`, entities are automatically extracted from the text using NLP and added to the `any_of` filter by default. This behavior can introduce unintended results, especially when using common entities, like places. Set to **`False`** (**recommended**) to retain full control over entity-based filtering. |
| `timestamp` | Time range: `start` and `end` in ISO 8601 format (e.g. `YYYY-MM-DDTHH:MM:SSZ`). Only content whose publication time falls in this range is counted. **Required for most volume queries.** |
| `entity` | Filter by entity IDs. Use `any_of`, `all_of`, or `none_of` with a list of IDs. Optional `search_in`: **HEADLINE** (only in headlines), **BODY** (only in body text), or **ALL** (both). Defaults to **ALL** if not provided. Example: `"entity": { "search_in": "HEADLINE", "any_of": ["D8442A"] }`. |
| `keyword` | Literal keyword matching with `any_of`, `all_of`, or `none_of`. Same optional `search_in` as `entity`: **HEADLINE**, **BODY**, or **ALL**. Defaults to **ALL** if not provided. |
| `source` | Restrict to a specific source (publisher) by source ID. |
| `sentiment` | Filter by the chunk's overall sentiment score from **-1.0** (negative) to **1.0** (positive). Chunk-level sentiment, not entity-specific. Multiple ranges supported (e.g. negative OR positive). |
| `document_type` | For the challenge, only **news** is available; use this to restrict to news documents. |

**Other filters:** The **topic** filter restricts to chunks where a specific event from the RavenPack Analytics taxonomy was detected. It supports the same optional `search_in`: **HEADLINE**, **BODY**, or **ALL** (defaults to **ALL**). For most general queries, the topic filter is not recommended.

---

### Ranking parameters

| Parameter | Description |
|-----------|-------------|
| `freshness_boost` | **Must be set to 0** for the challenge (and for point-in-time / historical research). Otherwise it boosts more recent documents; for 2021–2022 backtesting, set to **0** so time is not weighted. |

---

### Important notes

- **timestamp:** Use ISO 8601 format (e.g. `YYYY-MM-DDTHH:MM:SSZ`). Required for meaningful time-series volume.
- **entity / keyword / topic:** All three support the optional `search_in` parameter: **HEADLINE** (only in headlines), **BODY** (only in body text), or **ALL** (both). Defaults to **ALL** if not provided.
- **auto_enrich_filters:** Recommended value is **False** so you control entity filtering.
- **freshness_boost:** Recommended value is **0** to ignore publishing time (essential for point-in-time research).
- **Parameter validation:** The provider APIs do not have full coverage for parameter validation; typos or invalid values can lead to unexpected behavior. For full schemas and options, see the [official API documentation](https://docs.bigdata.com).

In [ ]:
volume_query = {
    "query": {
        "text": TEXT,
        "auto_enrich_filters": False,
        "filters": {
            "timestamp": {
                "start": f"{START_DATE}T00:00:00Z",
                "end": f"{END_DATE}T23:59:59Z"
            }
        },
        "ranking_params": {
            "freshness_boost": 0
        }
    }
}

response = session.post(VOLUME_ENDPOINT, json=volume_query)
try:
    volume_data = response.json()
except Exception:
    print(f"❌ Non-JSON response (status={response.status_code})")
    print(response.text[:1000] if response.text else "(empty body)")
    raise
print_volume_results(response, volume_data)

You'll see totals (documents, chunks) and per-day volume—filtered by your query and time range. Try changing the query text (`TEXT`), date range (`START_DATE`, `END_DATE`), or adding an entity filter and re-running the cell above to see how totals and per-day volume change.

### 1.1 Response structure

Understanding what the Volume API returns.

In short: you get totals (documents, chunks) and a per-day breakdown (volume and sentiment) for time-series plots.

**Response contains three main sections:**

**1. `results.total`** — Aggregates over the full date range
- `documents`: Total number of documents matching the query and filters
- `chunks`: Total number of chunks (text segments) across those documents

**2. `results.volume`** — Per-day breakdown (list of daily buckets)
- `date`: Day (YYYY-MM-DD)
- `documents`: Number of documents that have at least one chunk on that day
- `chunks`: Number of chunks on that day
- `sentiment`: (When available) aggregate sentiment for that day (e.g. for visualization)

**3. `usage`** — API consumption
- `api_query_units`: Credits consumed by this request

Use `results.volume` for time-series plots and `results.total` for quick overall size checks.

## 2. Volume evolution visualization

Next, we visualize **volume evolution** over time—how coverage and sentiment change day by day. Once you have per-day volume (and sentiment) from the Volume API, you can plot how coverage and sentiment evolve over time. This is useful for **theme or aggregated entity analysis** (e.g. when did semiconductor-related coverage spike?) and for **deciding where to run deeper Search API** queries.

Below we use the same query and date range but **filter by entity** (e.g. Apple) so the chart shows volume and sentiment for content that mentions that entity. The plot uses 7-day moving averages to smooth daily noise.

While this endpoint can be used to extract signals related to specific entities, it's important to note that obtaining data for many entities individually can be expensive, since the Volume API does not return a breakdown by entity within a single query. As a result, this API is best used for analyzing broader topics or collections (such as sectors or industry baskets), rather than large-scale entity-level signals.

In [ ]:
from api_helpers import plot_volume_evolution

# Query volume filtered by entity
volume_entity_query = {
    "query": {
        "text": TEXT,
        "auto_enrich_filters": False,
        "filters": {
            "timestamp": {
                "start": f"{START_DATE}T00:00:00Z",
                "end": f"{END_DATE}T23:59:59Z"
            },
            "entity": {"any_of": [ENTITY_ID]}
        },
        "ranking_params": {
            "freshness_boost": 0
        }
    }
}

response = session.post(VOLUME_ENDPOINT, json=volume_entity_query)
try:
    volume_entity_data = response.json()
except Exception:
    print(f"❌ Non-JSON response (status={response.status_code})")
    print(response.text[:1000] if response.text else "(empty body)")
    raise

fig_volume = plot_volume_evolution(
    volume_data=volume_entity_data,
    text=TEXT,
    start_date=START_DATE,
    end_date=END_DATE
)
if fig_volume:
    fig_volume.show()

## Summary and next steps

**What we covered**
- Volume API call with text query and time (and optional entity) filters
- Options (query, filters, ranking) and response structure (totals, per-day volume)
- Volume evolution visualization (per-day volume and sentiment plots)

**Next steps**
- **Search API** ([Search_API](../Search_API/)) — semantic document search with entity and keyword filters
- **CoMentions API** ([CoMentions_API](../CoMentions_API/)) — entity co-occurrence and relationship discovery
- **Knowledge Graph API** ([Knowledge_Graph_API](../Knowledge_Graph_API/)) — explore entities and relationships
- **Full API reference:** [docs.bigdata.com](https://docs.bigdata.com)

You can experiment by changing `TEXT`, `START_DATE`, `END_DATE`, or adding an entity filter and re-running the notebook to explore different volume queries.